[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyTorch and TensorFlow, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)

# Saving and Loading Models


## What you will be able to do

Save a trained model to a file in PyTorch and in Keras, load it back into a new process with the same
predictions, and save enough with it to carry on training where it stopped.

## The idea

### The problem

The station network in these notebooks trains in a few seconds. A real one can train for hours, and
everything it learned lives in the memory of one Python process. Colab ends a runtime that has sat
idle, a notebook's kernel restarts, and a program that should use the model to answer a question is a
different process from the one that trained it. Without a file, each of those means training again,
and unless every seed was fixed, training again gives a different model.

Saving is easy to get almost right. A file that holds the weights but not the optimizer's state
starts a resumed run from a different place, and a file whose loader runs code is a risk to anyone
who opens a copy from somewhere else.

### A formal definition

> A **checkpoint** is a file holding what a model has learned, its weights, and often what training
> needs in order to carry on: the optimizer's state and the step it reached. PyTorch saves a model's
> **state dict**, a dictionary from the name of each parameter, and of each buffer such as a batch
> norm layer's running mean, to its tensor. Keras saves a `.keras` file, a zip archive of the
> model's architecture, its weights, its optimizer's state and the settings it was compiled with.

### Why it works that way

PyTorch saves values and leaves the code to you. A state dict is names and tensors, so loading one
means building the model again from your own code and copying the tensors into it, and the names and
shapes have to match. `torch.save` can also store a whole model object with Python's `pickle`, but
loading a pickle can run any code the file names, so since PyTorch 2.6 `torch.load` loads only
tensors and plain containers unless told otherwise.

Keras saves the architecture too, as a description of every layer and its settings, so
`load_model` builds the model again without your code. The exception is a layer you wrote, whose
class Keras can find only if it was registered under a name.

The optimizer is part of training. Adam keeps two running averages for every weight, and a run that
loads the weights and starts a new optimizer takes different steps from one that never stopped. A
checkpoint for resuming holds both.

### Where this shows up

The **A Complete Project** notebook saves its classifier this way and loads it to answer for a new
station, and the **Overfitting and Regularization** notebook keeps the best weights seen during
training, in memory, which is the same state dict without the file.

### What this notebook covers

- what a state dict holds, and saving and loading one
- a checkpoint that resumes training exactly
- a `.keras` file, and Keras's weights on their own
- a layer of your own, registered so that Keras can load it
- four errors from saving and loading

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import torch

model = torch.nn.Linear(2, 1)
torch.save(model.state_dict(), "weights.pt")
again = torch.nn.Linear(2, 1)
again.load_state_dict(torch.load("weights.pt"))
print(torch.equal(again.weight, model.weight), torch.equal(again.bias, model.bias))
```

```
True True
```

`again` starts with random weights of its own, and loading the file replaces them with `model`'s.

## Setup

Three libraries, a way to quiet TensorFlow while it loads, the stations, and a folder for the files.

- `numpy` builds the stations
- `torch` is PyTorch
- `tensorflow` is TensorFlow, and `keras` its library of layers and training tools
- `contextlib` and `os` hide the messages TensorFlow prints while it loads, as in the **Why PyTorch
  and TensorFlow** notebook
- `Path` names the files, and `shutil` removes their folder at the end
- `zipfile` lists what is inside a `.keras` file

`station_network_arrays` makes the 1,200 made-up weather stations of the **Why PyTorch and
TensorFlow** notebook, from a generator started from a fixed seed, and the cell splits them into
training, validation and test stations and standardizes their four features with the training
stations' means. Every file the notebook writes goes in `scratch`, which the last cell removes.

In [1]:
import contextlib
import os
import shutil
import zipfile
from pathlib import Path

import numpy as np
import torch


@contextlib.contextmanager
def c_messages_hidden():
    """Send what C and C++ code writes to the error stream to os.devnull while the block runs."""
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 2)
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(devnull)
        os.close(saved)


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
with c_messages_hidden():
    import tensorflow as tf
    import keras


def station_network_arrays(n=1200, seed=2025):
    """The made-up stations of the scikit-learn guide, as arrays: four features and two targets."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    shift = np.where(coast_km < 3, 0.8,
            np.where(elevation > 700, -0.5,
            np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), 0.3, 0.0)))
    rng.choice(4, n, p=[0.4, 0.3, 0.1, 0.2])   # each station's owner, which this guide leaves out
    rng.integers(1950, 2025, n)                 # and the year it was installed
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    features = np.column_stack([latitude.round(2), longitude.round(2), elevation.round(0), coast_km.round(1)])
    return features.astype("float32"), jan_mean.round(1).astype("float32"), (may_mean < 2.5).astype("float32")


FEATURES = ["latitude", "longitude", "elevation_m", "coast_km"]
features, jan_mean, snow_on_may_1 = station_network_arrays()
order = np.random.default_rng(0).permutation(len(features))
train_rows, valid_rows, test_rows = order[:800], order[800:1000], order[1000:]
mean, std = features[train_rows].mean(axis=0), features[train_rows].std(axis=0)
scaled = (features - mean) / std

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)

print("torch", torch.__version__.split("+")[0], "| tensorflow", tf.__version__, "| keras", keras.__version__,
      "|", len(features), "stations")

torch 2.11.0 | tensorflow 2.21.0 | keras 3.13.2 | 1200 stations


## Worked examples

### A network to save

`make_net` builds the network, so that the code that builds it is written once and can build it
again, which loading a state dict needs. `train` takes full-batch steps on the 800 training stations
and returns the last loss.

In [2]:
X_train = torch.from_numpy(scaled[train_rows])
y_train = torch.from_numpy(jan_mean[train_rows]).unsqueeze(1)
X_test = torch.from_numpy(scaled[test_rows])
y_test = torch.from_numpy(jan_mean[test_rows]).unsqueeze(1)
loss_fn = torch.nn.MSELoss()


def make_net():
    return torch.nn.Sequential(torch.nn.Linear(4, 16), torch.nn.ReLU(), torch.nn.Linear(16, 1))


def train(model, optimizer, steps):
    for _ in range(steps):
        optimizer.zero_grad()
        loss = loss_fn(model(X_train), y_train)
        loss.backward()
        optimizer.step()
    return loss.item()


torch.manual_seed(0)
net = make_net()
print("training loss", round(train(net, torch.optim.Adam(net.parameters(), lr=0.01), 300), 3))
with torch.no_grad():
    print("test loss", round(loss_fn(net(X_test), y_test).item(), 3))

training loss 0.829
test loss 0.986


### What a state dict holds

In [3]:
state = net.state_dict()
print(type(state).__name__)
for name, tensor in state.items():
    print(f"{name:9}", tuple(tensor.shape))

OrderedDict
0.weight  (16, 4)
0.bias    (16,)
2.weight  (1, 16)
2.bias    (1,)


An ordered dictionary with an entry for each tensor the network learned. The names come from where
each layer sits in the `Sequential`, `0` for the first `Linear` and `2` for the second, since the
`ReLU` at position 1 has nothing to learn. A model written as a class gets the names of its
attributes instead, such as `head.weight`.

### Saving it, and loading it into a new network

In [4]:
torch.save(net.state_dict(), SCRATCH / "station_net.pt")
print(sorted(p.name for p in SCRATCH.iterdir()))

loaded = torch.load(SCRATCH / "station_net.pt")
print(type(loaded).__name__, list(loaded))

rebuilt = make_net()
print(rebuilt.load_state_dict(loaded))
with torch.no_grad():
    print(torch.equal(rebuilt(X_test), net(X_test)))

['station_net.pt']
OrderedDict ['0.weight', '0.bias', '2.weight', '2.bias']
<All keys matched successfully>
True


`torch.load` gives back the dictionary, `load_state_dict` copies its tensors into `rebuilt` and
reports that every name matched, and the two networks make the same predictions for all 200 test
stations, to the last bit. `rebuilt` came from `make_net`, so a program that uses the file needs that
function too.

### A checkpoint that resumes training

Three runs of the same network from the same seed. The first trains for 200 steps without stopping.
The second trains for 100, saves a checkpoint with the model, the optimizer and the step, and a new
network and optimizer load it and train for 100 more. The third loads only the model from that
checkpoint, and starts a new optimizer.

In [5]:
torch.manual_seed(0)
straight = make_net()
straight_loss = train(straight, torch.optim.Adam(straight.parameters(), lr=0.01), 200)

torch.manual_seed(0)
first_half = make_net()
first_optimizer = torch.optim.Adam(first_half.parameters(), lr=0.01)
train(first_half, first_optimizer, 100)
torch.save({"model": first_half.state_dict(), "optimizer": first_optimizer.state_dict(), "step": 100},
           SCRATCH / "checkpoint.pt")

checkpoint = torch.load(SCRATCH / "checkpoint.pt")
print(list(checkpoint), "| step", checkpoint["step"])

resumed = make_net()
resumed.load_state_dict(checkpoint["model"])
resumed_optimizer = torch.optim.Adam(resumed.parameters(), lr=0.01)
resumed_optimizer.load_state_dict(checkpoint["optimizer"])
resumed_loss = train(resumed, resumed_optimizer, 100)

model_only = make_net()
model_only.load_state_dict(checkpoint["model"])
fresh_optimizer_loss = train(model_only, torch.optim.Adam(model_only.parameters(), lr=0.01), 100)

print("without stopping       ", round(straight_loss, 3))
print("resumed with optimizer ", round(resumed_loss, 3))
print("resumed without it     ", round(fresh_optimizer_loss, 3))
print(all(torch.equal(a, b) for a, b in zip(straight.parameters(), resumed.parameters())),
      all(torch.equal(a, b) for a, b in zip(straight.parameters(), model_only.parameters())))

['model', 'optimizer', 'step'] | step 100
without stopping        0.958
resumed with optimizer  0.958
resumed without it      0.786
True False


The run resumed with its optimizer ends at the same loss as the run that never stopped, 0.958, and
every one of its weights is equal, `True`. The run that loaded only the model and started a new Adam
ends at 0.786, with different weights. Its loss happens to be lower, but it is a
different run from the one that was saved, and resuming is meant to continue that one.

### Keras saves the whole model

The same network in Keras, trained the same way, saved with `model.save` to a file whose name ends
in `.keras`.

In [6]:
keras.utils.set_random_seed(0)
net_keras = keras.Sequential([keras.Input((4,)), keras.layers.Dense(16, activation="relu"), keras.layers.Dense(1)])
net_keras.compile(optimizer=keras.optimizers.Adam(0.01), loss="mse")
net_keras.fit(scaled[train_rows], jan_mean[train_rows], epochs=300, batch_size=800, verbose=0)
print("test loss", round(net_keras.evaluate(scaled[test_rows], jan_mean[test_rows], verbose=0), 3))

net_keras.save(SCRATCH / "station_net.keras")
print(zipfile.ZipFile(SCRATCH / "station_net.keras").namelist())

test loss 0.976
['metadata.json', 'config.json', 'model.weights.h5']


The test loss is 0.976. The file is a zip archive: `config.json` describes the layers and the
settings the model was compiled with, and `model.weights.h5` holds their weights and the optimizer's
variables, beside a file of metadata.

In [7]:
loaded_keras = keras.models.load_model(SCRATCH / "station_net.keras")
print(np.array_equal(loaded_keras(scaled[test_rows]).numpy(), net_keras(scaled[test_rows]).numpy()))
print(type(loaded_keras.optimizer).__name__, "after", int(loaded_keras.optimizer.iterations.numpy()), "steps")

True
Adam after 300 steps


Loaded with no code that built it, the model makes the same predictions as `net_keras`, and it came
back compiled, with its Adam optimizer at the 300 steps it had taken.

### Keras's weights on their own

`save_weights` writes the weights without the architecture, to a file whose name must end in
`.weights.h5`, and `load_weights` copies them into a model built again from code, as PyTorch's state
dict does. A model that has trained also writes its optimizer's variables into that file, and a model
built without an optimizer loads the weights and leaves those unread.

In [8]:
net_keras.save_weights(SCRATCH / "station_net.weights.h5")

rebuilt_keras = keras.Sequential([keras.Input((4,)), keras.layers.Dense(16, activation="relu"), keras.layers.Dense(1)])
rebuilt_keras.load_weights(SCRATCH / "station_net.weights.h5")
print(np.array_equal(rebuilt_keras(scaled[test_rows]).numpy(), net_keras(scaled[test_rows]).numpy()))

True


A model built in code, with only the weights loaded into it, makes the same predictions again.

### A registered layer of your own

`Scale` multiplies its input by a fixed factor. For Keras to load a model that uses it, the class is
registered with the decorator `register_keras_serializable`, which records the class under a name
that the file stores. Keras also has to know the settings `__init__` was given. It records them by
itself when they are plain numbers, strings, booleans or `None`, as `factor` is, and `get_config`
writes them out explicitly, which a layer must do once one of its settings is anything else, such as
a NumPy array. `Scale` writes `get_config` even so, which shows what the file records and keeps the
layer saving if a setting of another kind is added.

In [9]:
@keras.saving.register_keras_serializable(package="guide")
class Scale(keras.layers.Layer):
    def __init__(self, factor=2.0, **kwargs):
        super().__init__(**kwargs)
        self.factor = factor

    def call(self, x):
        return x * self.factor

    def get_config(self):
        return {**super().get_config(), "factor": self.factor}


halved = keras.Sequential([keras.Input((4,)), Scale(0.5), keras.layers.Dense(1)])
halved.save(SCRATCH / "halved.keras")
reloaded = keras.models.load_model(SCRATCH / "halved.keras")
print(type(reloaded.layers[0]).__name__, reloaded.layers[0].factor)

Scale 0.5


The loaded model's first layer is a `Scale` again, with the factor of 0.5 that `get_config` saved and
`__init__` was given back.

## Your turn

Four tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/14-saving-and-loading-models-solutions.ipynb).

**1.** Print the names and shapes in the state dict of `torch.nn.Sequential(torch.nn.Linear(4, 8), torch.nn.ReLU(), torch.nn.Linear(8, 8), torch.nn.ReLU(), torch.nn.Linear(8, 1))`, and say why the numbers in the names skip.

In [10]:
# your code here

**2.** Save `net`'s state dict to `scratch/station_net.pt`, load the file into a new network from `make_net`, then add 1 to that network's first weight. Load the file again and show that it still holds the old value.

In [11]:
# your code here

**3.** Train a new network from `torch.manual_seed(0)` for 100 steps, save a checkpoint to `scratch/checkpoint.pt` that also holds the last training loss, and load it and print the step and the loss.

In [12]:
# your code here

**4.** Save `net_keras` to `scratch/copy.keras`, load it, train the copy for 10 more epochs with `fit`, and print its optimizer's step count before and after.

In [13]:
# your code here

## Common errors

### `RuntimeError: Error(s) in loading state_dict for Sequential:`

The saved state dict, loaded into a network with 32 hidden units where `net` had 16.

In [14]:
wider = torch.nn.Sequential(torch.nn.Linear(4, 32), torch.nn.ReLU(), torch.nn.Linear(32, 1))
wider.load_state_dict(torch.load(SCRATCH / "station_net.pt"))

RuntimeError: Error(s) in loading state_dict for Sequential:
	size mismatch for 0.weight: copying a param with shape torch.Size([16, 4]) from checkpoint, the shape in current model is torch.Size([32, 4]).
	size mismatch for 0.bias: copying a param with shape torch.Size([16]) from checkpoint, the shape in current model is torch.Size([32]).
	size mismatch for 2.weight: copying a param with shape torch.Size([1, 16]) from checkpoint, the shape in current model is torch.Size([1, 32]).

Every name matched, and three of the four shapes did not, and the message lists each one with the
saved shape first and the model's second. A state dict loads only into a model built the way the
saved one was, which is the reason for a function such as `make_net`.

### `UnpicklingError: Weights only load failed.`

The whole network saved with `torch.save(net)`, then loaded.

In [15]:
torch.save(net, SCRATCH / "whole_net.pt")
torch.load(SCRATCH / "whole_net.pt")

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL torch.nn.modules.container.Sequential was not an allowed global by default. Please use `torch.serialization.add_safe_globals([torch.nn.modules.container.Sequential])` or the `torch.serialization.safe_globals([torch.nn.modules.container.Sequential])` context manager to allowlist this global if you trust this class/function.

Check the documentation

`torch.save(net)` pickled the `Sequential` object, and `torch.load` refuses to unpickle a class it
was not told to trust, because loading a pickle can run code. The message names the class it met,
`torch.nn.modules.container.Sequential`. Save the state dict instead, as the worked examples do.
`weights_only=False` loads the object, and is safe only for a file you made yourself.

In [16]:
print(type(torch.load(SCRATCH / "whole_net.pt", weights_only=False)).__name__)

Sequential


### `ValueError: Invalid filepath extension for saving.`

A Keras model saved to a name with no extension.

In [17]:
net_keras.save(SCRATCH / "station_net")

ValueError: Invalid filepath extension for saving. Please add either a `.keras` extension for the native Keras format (recommended) or a `.h5` extension. Use `model.export(filepath)` if you want to export a SavedModel for use with TFLite/TFServing/etc. Received: filepath=scratch/station_net.

`model.save` chooses the format from the name, and refuses one it does not know. Give the file the
`.keras` ending, the native format the message recommends.

### `TypeError: <class 'keras.src.models.sequential.Sequential'> could not be deserialized properly.`

A model with a layer like `Scale` but not registered, saved and loaded. The error's full message
prints the model's whole configuration, including a Python object id that differs on every run, so
the cell catches the error and prints the first sentence of the error and of the error that caused it.

In [18]:
class Shift(keras.layers.Layer):
    def __init__(self, amount=1.0, **kwargs):
        super().__init__(**kwargs)
        self.amount = amount

    def call(self, x):
        return x + self.amount

    def get_config(self):
        return {**super().get_config(), "amount": self.amount}


keras.Sequential([keras.Input((4,)), Shift(), keras.layers.Dense(1)]).save(SCRATCH / "shifted.keras")
try:
    keras.models.load_model(SCRATCH / "shifted.keras")
except TypeError as error:
    print("TypeError:", str(error).split(". ")[0] + ".")
    cause = error.__cause__ or error.__context__
    print("caused by", type(cause).__name__ + ":", str(cause).split(". ")[0] + ".")

TypeError: <class 'keras.src.models.sequential.Sequential'> could not be deserialized properly.
caused by TypeError: Could not locate class 'Shift'.


The first sentence names the model, which is the object Keras was building when it failed. The
cause, printed second, names the layer it could not find. Register the class with
`@keras.saving.register_keras_serializable()`, or give `load_model` the class by its name.

In [19]:
fixed = keras.models.load_model(SCRATCH / "shifted.keras", custom_objects={"Shift": Shift})
print(type(fixed.layers[0]).__name__, fixed.layers[0].amount)

shutil.rmtree(SCRATCH)
print("scratch removed:", not SCRATCH.exists())

Shift 1.0
scratch removed: True


## Recap

- You can save a PyTorch model's state dict and load it into the same network built again from code.
- You can save a checkpoint with the optimizer and the step, and resume training exactly.
- You can save a Keras model to a `.keras` file and load it without its code, or save its weights
  alone.
- You can write a Keras layer that loads, with a registered name, and `get_config` for settings that
  are not plain values.

## What is next

The **A Complete Project** notebook puts the guide together, building, training, evaluating, saving
and loading one classifier in PyTorch and again in Keras.

---

&#8592; **Previous:** [Recurrent Networks](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/13-recurrent-networks.ipynb)  &nbsp;·&nbsp;  [PyTorch and TensorFlow, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [A Complete Project](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/15-a-complete-project.ipynb) &#8594;
